# Phase 1 — Understand FinanceBench and verify the dataset

This notebook is for students who have completed Phase 0 and are starting to use Python. Select the **Financial RAG** kernel. Ollama is not needed for this lesson.

Complete **Phase 1**, explain the verification results, and then confirm whether to begin **Phase 2**. Do not select Run All. Phase 2 has an explicit stop gate.

You will learn how JSONL stores questions, how metadata determines the subset, why complete PDFs are the retrieval source, and how page numbers, years, units and table extraction affect answers.

The existing data has already been downloaded. This lesson mainly reads the source files and saves teaching figures and records to `rag_study/outputs/data_audit/`. It does not change the report, raw data or retrieval index.

**Lesson order:** use this notebook for Phase 1 only. Its optional Phase 2 cells are an earlier introduction; the current repair comparison, visual audit and worksheet are in [the dedicated Phase 2 notebook](phase_02_pdf_parsing_and_chunk_audit.ipynb). Keep the stop gate below unchanged.

## Phase 1 / Step 1: Locate the data

`Path` represents a file location. Each line in a JSONL file is one JSON record. `read_jsonl` converts each line into a Python dictionary and collects the records in a list.

The virtual environment, model weights and benchmark serve different purposes: Python runs the code, the model generates answers, and the benchmark supplies questions, reference answers and reports.

In [ ]:
from pathlib import Path
import json
import hashlib
import sys
import pandas as pd
import matplotlib.pyplot as plt
from pypdf import PdfReader

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "rag_study/config.json").exists())
DATA = ROOT / "rag_study/data"
OUT = ROOT / "rag_study/outputs/data_audit"
OUT.mkdir(parents=True, exist_ok=True)

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

print("Python:", sys.executable)
print("Data folder:", DATA)
assert Path(sys.prefix).resolve() == (ROOT / "rag_study/.venv").resolve(), "Choose Financial RAG."

## Step 2: Read the source data and one development question

`questions` contains 150 public questions, including reference answers and gold evidence. `documents` is the document metadata table. Its 361 records do not mean that 361 PDFs have been downloaded.

We display only a development question. Do not read evaluation answers to adjust prompts during development. Never put `answer`, `justification` or `evidence` into the retrieval index or model prompt.

In [ ]:
questions = read_jsonl(DATA / "source/financebench_open_source.jsonl")
document_rows = read_jsonl(DATA / "source/financebench_document_information.jsonl")
documents = {d["doc_name"]: d for d in document_rows}
saved_dev = read_jsonl(DATA / "dev.jsonl")
saved_test = read_jsonl(DATA / "test.jsonl")
manifest = pd.read_csv(DATA / "manifest.csv")
provenance = json.loads((DATA / "source/provenance.json").read_text())

print("Public questions:", len(questions))
print("Document metadata records:", len(document_rows))
print("Question fields:", list(questions[0]))
print("Document fields:", list(document_rows[0]))
print("Pinned revision:", provenance["revision"])
q = saved_dev[0]
display(pd.DataFrame([{k: q[k] for k in ["financebench_id", "company", "doc_name", "question"]}]))

## Step 3: Reproduce the counts: 52 / 18 / 10 / 8

First, use `doc_name` to look up each question's document. Evaluation records must satisfy `doc_type == "10k"` and `doc_period == 2022`, with all annotated evidence contained in the report named by the question.

For development, select questions from earlier 10-K reports and companies outside the evaluation universe. Sort them by `financebench_id` and take the first ten. This is not a random split, and there is no model-training set.

In [ ]:
def evidence_in_named_report(q):
    return bool(q["evidence"]) and all(
        e.get("doc_name", e.get("evidence_doc_name")) == q["doc_name"]
        for e in q["evidence"]
    )

test = sorted([
    q for q in questions
    if documents[q["doc_name"]]["doc_type"] == "10k"
    and int(documents[q["doc_name"]]["doc_period"]) == 2022
    and evidence_in_named_report(q)
], key=lambda q: q["financebench_id"])
test_companies = {q["company"] for q in test}
dev = sorted([
    q for q in questions
    if documents[q["doc_name"]]["doc_type"] == "10k"
    and int(documents[q["doc_name"]]["doc_period"]) < 2022
    and q["company"] not in test_companies
    and evidence_in_named_report(q)
], key=lambda q: q["financebench_id"])[:10]

assert len(questions) == 150
assert len(test) == 52 and len({q["doc_name"] for q in test}) == 18
assert len(dev) == 10 and len({q["doc_name"] for q in dev}) == 8
assert test == saved_test and dev == saved_dev
assert {q["company"] for q in dev}.isdisjoint(test_companies)
assert {q["financebench_id"] for q in dev}.isdisjoint({q["financebench_id"] for q in test})

display(pd.DataFrame([
    {"split": "development", "questions": len(dev), "reports": len({q["doc_name"] for q in dev})},
    {"split": "evaluation", "questions": len(test), "reports": len({q["doc_name"] for q in test})},
]))

## Step 4: Verify the 26 complete PDFs

The manifest is a file inventory containing original URLs, download URLs, fiscal years, file sizes, page counts and SHA-256 hashes. SHA-256 acts as a fingerprint of file contents. Matching hashes mean that the file bytes match; they do not prove that extraction is correct.

This step reads the actual page counts of all PDFs without evaluating test answers. `evidence_page_num` is a zero-based index and must fall within the document's page range.

In [ ]:
report_names = {q["doc_name"] for q in dev + test}
assert len(report_names) == 26
assert report_names == set(manifest["doc_name"])
assert report_names == {p.stem for p in (DATA / "pdfs").glob("*.pdf")}

for row in manifest.to_dict("records"):
    path = DATA / "pdfs" / f"{row['doc_name']}.pdf"
    raw = path.read_bytes()
    assert raw.startswith(b"%PDF-")
    assert len(raw) == row["bytes"]
    assert hashlib.sha256(raw).hexdigest() == row["sha256"], row["doc_name"]
    assert len(PdfReader(path).pages) == row["pdf_pages"], row["doc_name"]

pages_by_doc = manifest.set_index("doc_name")["pdf_pages"].to_dict()
for q in dev + test:
    assert all(0 <= e["evidence_page_num"] < pages_by_doc[q["doc_name"]]
               for e in q["evidence"])

verification = {
    "source_revision": provenance["revision"],
    "public_questions": len(questions), "evaluation_questions": len(test),
    "evaluation_reports": len({q["doc_name"] for q in test}),
    "development_questions": len(dev), "development_reports": len({q["doc_name"] for q in dev}),
    "pdf_count": len(report_names), "pdf_pages": int(manifest["pdf_pages"].sum()),
    "pdf_hashes_verified": True, "actual_pdf_page_counts_verified": True,
    "manual_document_audit_complete": False,
}
(OUT / "phase1_verification.json").write_text(json.dumps(verification, indent=2))
print("Verified all 26 PDFs; total pages:", verification["pdf_pages"])
display(manifest[["doc_name", "doc_period", "split", "pdf_pages", "bytes"]].head())

## Step 5: Plot the question distribution

Each bar shows the number of evaluation questions associated with one report. Unequal bar lengths mean that companies or reports contribute different numbers of questions. This does not demonstrate model accuracy. The figure is suitable for the dataset description in Section 2.2. The existing report already includes a similar figure, so there is no need to add a duplicate.

In [ ]:
counts = pd.Series([q["doc_name"] for q in test]).value_counts().sort_values()
fig, ax = plt.subplots(figsize=(9, 7))
counts.plot.barh(ax=ax, color="#42657d")
ax.set(title="FinanceBench evaluation subset: 52 questions / 18 reports",
       xlabel="Number of questions", ylabel="Report")
ax.xaxis.get_major_locator().set_params(integer=True)
fig.tight_layout()
fig.savefig(OUT / "phase1_question_counts.png", dpi=180)
plt.show()

## Step 6: Plot complete-report page counts

This figure shows the lengths of the documents that retrieval must search. Development and evaluation reports have different colours. Longer documents may make evidence harder to find, but this figure alone cannot establish a causal relationship.

Use it to supplement the document inventory. If space in the report is limited, keep it in the notebook or appendix.

In [ ]:
lengths = manifest.sort_values(["pdf_pages", "doc_name"])
colors = lengths["split"].map({"dev": "#b87b40", "test": "#42657d"})
fig, ax = plt.subplots(figsize=(9, 9))
ax.barh(lengths["doc_name"], lengths["pdf_pages"], color=colors)
ax.set(title="26 complete annual reports", xlabel="PDF pages", ylabel="Report")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#b87b40", label="Development"),
                   Patch(color="#42657d", label="Evaluation")])
fig.tight_layout()
fig.savefig(OUT / "phase1_report_pages.png", dpi=180)
plt.show()

## Phase 1 exercises and stopping point

Answer these questions in your own words:
1. Why do we use only 62 of the 150 public questions, and why do those 62 questions require only 26 PDFs?
2. Why do 361 metadata records not mean that 361 documents have been downloaded?
3. Does a successful hash check prove that table units were extracted correctly?

You should now be able to locate the original questions, metadata, splits, PDFs and manifest. **Summarise Phase 1, then confirm before starting Phase 2.**

There is no need to download the data again for this lesson. If a file is missing, check its path first. Then use the existing command `rag_study/.venv/bin/python rag_study/acquire.py`. This command needs network access, downloads missing files using the saved revision, and rewrites the splits and manifest.

1. **We select 62 of the 150 public questions because the study has a fixed scope.** The evaluation set includes only FY2022 Form 10-K reports, and every question's annotated evidence must be contained in its named report. This gives **52 questions from 18 reports**. The development set contains **10 questions from earlier reports and companies outside the evaluation set, covering 8 reports**. One report can support several questions, so the 62 questions require only **26 complete PDFs**.

2. **The 361 metadata records describe documents; they do not count downloaded files.** Each record describes a document's name, company, year, type and source URL. We download only the **26 reports** selected for this study. A table with 361 records does not establish that 361 files were downloaded.

3. **No. A successful hash check only shows that the PDF contents match the file recorded in the manifest.** It does not prove that parsed tables are correct. For example, an intact source PDF may still yield extracted text that omits the unit “millions” or assigns numbers to the wrong year. Phase 2 therefore requires comparison with the original pages.

All data locations are under `rag_study/data/`:

| Data | Location |
|---|---|
| Original questions | `source/financebench_open_source.jsonl` |
| Document metadata | `source/financebench_document_information.jsonl` |
| Development/evaluation splits | `dev.jsonl`, `test.jsonl` |
| Complete PDFs | `pdfs/` |
| File inventory | `manifest.csv`, `manifest.json` |

Phase 1 verifies the selection rules, file integrity and page counts. **Whether the extracted financial content is correct remains a Phase 2 check.**

# Phase 2: Compare source pages, text, tables and chunks

The 13-page extraction audit has been completed and author-confirmed; this section is an optional explanation of that audit, not a new grading requirement. Successfully extracting text does not mean that years, numbers and units remain correctly aligned. Start with one page without rebuilding the full corpus or index.

The next cell defaults to `False`. Change it to `True` only after reviewing Phase 1 and confirming that you are ready to begin Phase 2.

**Current workflow:** stop here and open `phase_02_pdf_parsing_and_chunk_audit.ipynb`. The optional cells below are retained for reference; their worksheet is not the current revision-specific audit.

In [ ]:
CONTINUE_TO_PHASE_2 = False
assert CONTINUE_TO_PHASE_2, "Stop here: review Phase 1 before starting Phase 2."

## Step 1: Understand page numbering and open the source PDF

The first development question concerns Pfizer FY2021. Gold evidence page index 58 corresponds to page 59 in the PDF viewer. The page number printed on the document may differ.

`pdf.pages[58]` means the 59th object in a Python list. Open the PDF shown below, read that page, and note the relevant row, years and units. The gold page is used here for a development audit; it will not be supplied as a retrieval hint during the experiment.

In [ ]:
q = dev[0]
e = q["evidence"][0]
doc_name = q["doc_name"]
page0 = e["evidence_page_num"]
pdf_path = DATA / "pdfs" / f"{doc_name}.pdf"
print("Question:", q["question"])
print("PDF:", pdf_path)
print("Python page index:", page0, "| PDF viewer page:", page0 + 1)

## Step 2: Extract one page only

`extract_text(layout=True)` attempts to preserve text positions. `extract_tables()` returns rows and cells. Compare the results with the source PDF: check column years, monetary units, negative values shown in parentheses, and merged cells. Displaying the first eight rows is only an inspection aid; it does not mean that the whole table has been checked.

In [ ]:
import pdfplumber
with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[page0]
    page_text = page.extract_text(layout=True) or ""
    tables = page.extract_tables()
print(page_text[:1800])
print("Detected tables:", len(tables))
if tables:
    display(pd.DataFrame(tables[0]).head(8))
else:
    print("No detected table: inspect the text and PDF; record the limitation.")

## Step 3: Compare the saved page and chunk

A parsed record stores extraction results for a complete page. A chunk is a smaller passage supplied to the retriever. Its ID includes the source document and PDF page so that citations can lead back to the source.

Do not insert reference answers into chunks. The 350/50 chunk settings use the embedding tokenizer. The later 2,000-token evidence budget uses the generator tokenizer. These token counts are not interchangeable.

In [ ]:
parsed_pages = read_jsonl(DATA / "parsed" / f"{doc_name}.jsonl")
page_record = next(p for p in parsed_pages if p["page0"] == page0)
assert page_record["pdf_page"] == page0 + 1
assert page_record["text"] == page_text, "Extraction differs: record parser/runtime changes."
chunks = read_jsonl(DATA / "chunks.jsonl")
page_chunks = [c for c in chunks if c["doc_name"] == doc_name and c["page0"] == page0]
assert all(c["pdf_page"] == page0 + 1 for c in page_chunks)
display(pd.DataFrame([{k: c[k] for k in ["id", "kind", "embedding_tokens"]}
                     for c in page_chunks]))
if page_chunks:
    print("First chunk:\n", page_chunks[0]["text"][:1500])

## Step 4: Inspect chunk lengths and recorded issues

The histogram shows chunk lengths in embedding tokens. A concentration near the target length helps check the splitting behaviour. You still need to read the text to check whether meaning and units remain intact.

Warnings flag items for review; they do not mean that every affected page failed to parse. Do not silently remove affected questions or use test reference answers to repair the parser.

In [ ]:
warnings = read_jsonl(DATA / "parsing_failures.jsonl")
print("Existing chunks:", len(chunks), "| Recorded warnings:", len(warnings))
display(pd.DataFrame(warnings))
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist([c["embedding_tokens"] for c in chunks], bins=30, color="#42657d")
ax.axvline(350, color="#b87b40", linestyle="--", label="350-token target")
ax.set(title="Existing corpus chunk lengths", xlabel="Embedding tokens", ylabel="Chunks")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "phase2_chunk_lengths.png", dpi=180)
plt.show()

## Step 5: Create a manual audit worksheet without automatically marking passes

The CSV has one row per development question/evidence-page pair. Record your checks and specific observations: how years and numbers align, whether units and negative values remain, whether merged cells cause ambiguity, and the printed page label.

`review_status` defaults to `pending`. Enter `not_applicable` where appropriate and explain why. Record uncertainty when you cannot decide. An existing audit worksheet will not be overwritten.

In [ ]:
audit_path = OUT / "development_document_audit.csv"
if not audit_path.exists():
    audit_rows = []
    for q in dev:
        for page0 in sorted({e["evidence_page_num"] for e in q["evidence"]}):
            audit_rows.append({
                "financebench_id": q["financebench_id"], "doc_name": q["doc_name"],
                "page0": page0, "pdf_page": page0 + 1, "printed_page_label": "",
                "page_mapping_ok": "", "years_and_columns_ok": "", "units_ok": "",
                "negative_values_ok": "", "headers_ok": "", "merged_cells_ok": "",
                "chunk_supports_required_facts": "", "review_status": "pending", "notes": "",
            })
    pd.DataFrame(audit_rows).to_csv(audit_path, index=False)
print("Fill this CSV after viewing the original pages:", audit_path)
display(pd.read_csv(audit_path).head())

## Phase 2 exercises, report material and stopping point

Choose another development question by setting `q = dev[1]`, then repeat the page comparison and chunk inspection. If it has several evidence pages, check each page.

Claim a “manual check” only after comparing the extraction with the original PDF. Sections 2.2–2.3 of the report can include a checked table example, the actual audit scope and unresolved extraction issues. Do not treat chunk counts as accuracy.

**Stop and summarise:** Which pages did you check? What unit or table issues did you find? How many CSV entries remain pending? Confirm before starting Phase 3. This notebook does not rebuild the corpus, query the model, freeze settings or run the final test.